# End-to-End PySpark Data Engineering Project


## Task 1: Data Ingestion & Exploration


In [2]:
# 1. Install Dependencies
!pip install  pyspark findspark openml pyarrow 

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.4/160.4 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.8/93.8 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 52.4 MB/s eta 0:00:00:00:01
  Created wheel for liac-arff: filename=liac_arff-2.5.0-py3-none-any.whl size=11717 sha256=2ad96beca1fdaeed6e1bc381b26e1fec99c0c1023cca1f70a0d3c4a9c8bb3c1c
  Stored in directory: /root/.cache/pip/wheels/a9/ac/cf/c2919807a5c623926d217c0a18eb5b457e5c19d242c3b5963a
Successfully built liac-arff


In [ ]:
#SparkSession: This is the universal entry point for programming Spark with the Dataset and DataFrame AP
from pyspark.sql import SparkSession # type: ignore
#These are specialized, highly optimized functions that execute on distributed Spark data nodes 
from pyspark.sql.functions import * # type: ignore
#from pyspark.sql.window import Window
from pyspark.sql.window import Window # type: ignore
from pyspark.sql.types import * # type: ignore
import pandas as pd # type: ignore


spark=SparkSession.builder\
     .appName("PySpark_Data_Engineering_Project") \
          .master("local[*]") \
              .config("spark.driver.memory", "4g") \
                  .getOrCreate()

## Google Drive Mount

In [4]:
from google.colab import drive # type: ignore
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
print("Loading dataset from drive...")
df = spark.read.csv(
    "/content/drive/MyDrive/BNPParibas_Data.csv",
    header=True,
    inferSchema=True
)

Loading dataset from drive...


In [6]:
df.show()

+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+
|customer_id|age|tenure_months|monthly_charges|total_charges| contract_type|internet_service|support_tickets|payment_method|churn|
+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+
|          1| 56|           15|          59.23|       929.62|Month-to-Month|           Fiber|              5|           UPI|    1|
|          2| 69|           64|           24.2|       1613.6|Month-to-Month|           Fiber|              1|   Credit Card|    1|
|          3| 46|           28|          78.02|      2053.63|      One Year|           Fiber|              1|           UPI|    0|
|          4| 32|           39|          45.95|      1688.47|Month-to-Month|           Fiber|              1|           UPI|    0|
|          5| 60|           57|          49.88|      2950.23|Month-to-Month|       

In [7]:
# Display Dataset Overview
print('Schema')
df.printSchema()


Schema
root
 |-- customer_id: integer (nullable = true)
 |-- age: integer (nullable = true)
 |-- tenure_months: integer (nullable = true)
 |-- monthly_charges: double (nullable = true)
 |-- total_charges: double (nullable = true)
 |-- contract_type: string (nullable = true)
 |-- internet_service: string (nullable = true)
 |-- support_tickets: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- churn: integer (nullable = true)



### Count

In [8]:
print(f"Record Count: {df.count()}") #type:ignore

Record Count: 1000


In [9]:
print("Summary Statistics:")
df.describe().show()

Summary Statistics:
+-------+-----------------+------------------+------------------+-----------------+------------------+--------------+----------------+------------------+--------------+------------------+
|summary|      customer_id|               age|     tenure_months|  monthly_charges|     total_charges| contract_type|internet_service|   support_tickets|payment_method|             churn|
+-------+-----------------+------------------+------------------+-----------------+------------------+--------------+----------------+------------------+--------------+------------------+
|  count|             1000|              1000|              1000|             1000|              1000|          1000|            1000|              1000|          1000|              1000|
|   mean|            500.5|            43.819|            35.459|79.96715000000002| 2800.235379999997|          NULL|            NULL|             1.956|          NULL|             0.502|
| stddev|288.8194360957494|14.9910296500

###  Null Count

In [10]:
df.select([count(when(col(c).isNull(),c)).alias(c)for c in df.columns]).show() #type:ignore

+-----------+---+-------------+---------------+-------------+-------------+----------------+---------------+--------------+-----+
|customer_id|age|tenure_months|monthly_charges|total_charges|contract_type|internet_service|support_tickets|payment_method|churn|
+-----------+---+-------------+---------------+-------------+-------------+----------------+---------------+--------------+-----+
|          0|  0|            0|              0|            0|            0|               0|              0|             0|    0|
+-----------+---+-------------+---------------+-------------+-------------+----------------+---------------+--------------+-----+



### Duplicate Count

In [11]:
duplicates = df.count() - df.dropDuplicates().count()
print(f"Duplicate Count: {duplicates}")

Duplicate Count: 0


### Data Types

In [12]:
print("Data Types :")
df.dtypes

Data Types :


[('customer_id', 'int'),
 ('age', 'int'),
 ('tenure_months', 'int'),
 ('monthly_charges', 'double'),
 ('total_charges', 'double'),
 ('contract_type', 'string'),
 ('internet_service', 'string'),
 ('support_tickets', 'int'),
 ('payment_method', 'string'),
 ('churn', 'int')]

# Production-Ready Data Pipeline (Bronze Layer)

In [13]:
import os

# Define the root path in your Google Drive where you want the project data to live
base_path = "/content/drive/MyDrive/pyspark_openml_project/data"

# Create the folders inside Google Drive
for layer in ['bronze', 'silver', 'gold']:
    folder = f"{base_path}/{layer}"
    os.makedirs(folder, exist_ok=True)
    
# Write to Bronze layer in Google Drive
df.write.mode("overwrite").parquet(f"{base_path}/bronze/raw_data.parquet")
print("Saved to Bronze layer in Google Drive.")


Saved to Bronze layer in Google Drive.


## Task 2: ETL Pipeline Development


In [14]:
# Extract from Bronze
df_bronze = spark.read.parquet(f"{base_path}/bronze/raw_data.parquet")

In [15]:
df_bronze.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- age: integer (nullable = true)
 |-- tenure_months: integer (nullable = true)
 |-- monthly_charges: double (nullable = true)
 |-- total_charges: double (nullable = true)
 |-- contract_type: string (nullable = true)
 |-- internet_service: string (nullable = true)
 |-- support_tickets: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- churn: integer (nullable = true)



In [16]:
# 1. Duplicate Removal
df_transformed = df_bronze.dropDuplicates()

In [17]:
# 2. Missing Value Treatment
num_cols = ['age', 'tenure_months', 'monthly_charges', 'total_charges', 'support_tickets']
df_transformed = df_transformed.fillna(0.0, subset=num_cols)
df_transformed.show()

+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+
|customer_id|age|tenure_months|monthly_charges|total_charges| contract_type|internet_service|support_tickets|payment_method|churn|
+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+
|          1| 56|           15|          59.23|       929.62|Month-to-Month|           Fiber|              5|           UPI|    1|
|        182| 23|           38|          80.86|      3246.77|Month-to-Month|             DSL|              3|          Cash|    1|
|        199| 20|           40|          29.91|      1133.23|      Two Year|           Fiber|              3|   Credit Card|    0|
|        410| 19|           51|          88.61|      4368.39|      One Year|           Fiber|              0|           UPI|    0|
|        664| 56|           47|         141.36|      6544.26|Month-to-Month|       

In [18]:
# 3. Data Type Conversion
df_transformed = df_transformed.withColumn("churn", col("churn").cast("int")) \
                               .withColumn("total_charges", col("total_charges").cast("double")) \
                               .withColumn("age", col("age").cast("integer"))
df_transformed.show()                               

+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+
|customer_id|age|tenure_months|monthly_charges|total_charges| contract_type|internet_service|support_tickets|payment_method|churn|
+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+
|          1| 56|           15|          59.23|       929.62|Month-to-Month|           Fiber|              5|           UPI|    1|
|        182| 23|           38|          80.86|      3246.77|Month-to-Month|             DSL|              3|          Cash|    1|
|        199| 20|           40|          29.91|      1133.23|      Two Year|           Fiber|              3|   Credit Card|    0|
|        410| 19|           51|          88.61|      4368.39|      One Year|           Fiber|              0|           UPI|    0|
|        664| 56|           47|         141.36|      6544.26|Month-to-Month|       

In [19]:
# 4. Feature Engineering
# Create a new feature 'is_senior' based on age
df_transformed = df_transformed.withColumn("is_senior", when(col("age") >= 60, 1).otherwise(0))
# Create 'charge_per_tenure' to see how much they pay relative to their tenure
df_transformed = df_transformed.withColumn("charge_per_tenure", round(col("total_charges") / (col("tenure_months") + 1),3))

In [20]:
# 5. Aggregation
# Calculate average monthly charges by contract type and join it back to the main DataFrame
contract_avg_df = df_transformed.groupBy("contract_type").agg(avg("monthly_charges").alias("avg_contract_monthly_charges"))
contract_avg_df.show()

+--------------+----------------------------+
| contract_type|avg_contract_monthly_charges|
+--------------+----------------------------+
|Month-to-Month|           80.42109777015438|
|      One Year|           80.04780575539571|
|      Two Year|           77.90187050359717|
+--------------+----------------------------+



In [21]:
df_transformed.show()

+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+---------+-----------------+
|customer_id|age|tenure_months|monthly_charges|total_charges| contract_type|internet_service|support_tickets|payment_method|churn|is_senior|charge_per_tenure|
+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+---------+-----------------+
|          1| 56|           15|          59.23|       929.62|Month-to-Month|           Fiber|              5|           UPI|    1|        0|           58.101|
|        182| 23|           38|          80.86|      3246.77|Month-to-Month|             DSL|              3|          Cash|    1|        0|           83.251|
|        199| 20|           40|          29.91|      1133.23|      Two Year|           Fiber|              3|   Credit Card|    0|        0|            27.64|
|        410| 19|           51|          88.61

# Load Store transformed data in:
## silver_layer/



In [22]:
df_transformed.write.mode("overwrite").parquet(f"{base_path}/silver/cleaned_data.parquet")
print("Saved to Silver layer. Here is a preview of the transformed data:")

Saved to Silver layer. Here is a preview of the transformed data:


## Task 3: ELT Pipeline & Medallion Architecture (Gold Layer)


In [23]:
# Read from Silver
df_silver = spark.read.parquet(f"{base_path}/silver/cleaned_data.parquet")

In [24]:
df_silver.show()

+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+---------+-----------------+
|customer_id|age|tenure_months|monthly_charges|total_charges| contract_type|internet_service|support_tickets|payment_method|churn|is_senior|charge_per_tenure|
+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+---------+-----------------+
|          1| 56|           15|          59.23|       929.62|Month-to-Month|           Fiber|              5|           UPI|    1|        0|           58.101|
|        182| 23|           38|          80.86|      3246.77|Month-to-Month|             DSL|              3|          Cash|    1|        0|           83.251|
|        199| 20|           40|          29.91|      1133.23|      Two Year|           Fiber|              3|   Credit Card|    0|        0|            27.64|
|        410| 19|           51|          88.61

In [25]:
#Generate minimum 3 business KPIs.
#1. Churn Distribution
kpi1_churn_dist = df_silver.groupBy("churn").count().withColumnRenamed("count", "total_customers")
kpi1_churn_dist.show()

+-----+---------------+
|churn|total_customers|
+-----+---------------+
|    1|            502|
|    0|            498|
+-----+---------------+



In [26]:
# 2. Average Monthly Charges by Churn
kpi2_avg_charges = df_silver.groupBy("churn").agg(avg("monthly_charges").alias("avg_monthly_charges"))
kpi2_avg_charges.show()

+-----+-------------------+
|churn|avg_monthly_charges|
+-----+-------------------+
|    1|  86.39589641434263|
|    0|  73.48676706827304|
+-----+-------------------+



In [27]:
# 3. Churn by Contract Type
kpi3_contract_churn = df_silver.groupBy("contract_type", "churn").count()
kpi3_contract_churn.show()

+--------------+-----+-----+
| contract_type|churn|count|
+--------------+-----+-----+
|      Two Year|    1|   30|
|      One Year|    0|  211|
|Month-to-Month|    1|  405|
|      Two Year|    0|  109|
|Month-to-Month|    0|  178|
|      One Year|    1|   67|
+--------------+-----+-----+



In [28]:
# Save to Gold
kpi1_churn_dist.write.mode("overwrite").parquet(f"{base_path}/gold/kpi1_churn_dist.parquet")
kpi2_avg_charges.write.mode("overwrite").parquet(f"{base_path}/gold/kpi2_avg_charges.parquet")
kpi3_contract_churn.write.mode("overwrite").parquet(f"{base_path}/gold/kpi3_contract_churn.parquet")
print("Saved KPIs to Gold layer.")

Saved KPIs to Gold layer.


# Task 4: PySpark + Pandas Integration

In [29]:
# Convert Spark to Pandas
df_pandas=df_silver.toPandas()

# Feature Engineering in Pandas
df_pandas['Average_monthly_charge']=(df_pandas['total_charges']/ (df_pandas['tenure_months']+1)).round(2)
df_pandas.head(10)


,customer_id,age,tenure_months,monthly_charges,total_charges,contract_type,internet_service,support_tickets,payment_method,churn,is_senior,charge_per_tenure,Average_monthly_charge
0,1,56,15,59.23,929.62,Month-to-Month,Fiber,5,UPI,1,0,58.101,58.10
1,182,23,38,80.86,3246.77,Month-to-Month,DSL,3,Cash,1,0,83.251,83.25
2,199,20,40,29.91,1133.23,Two Year,Fiber,3,Credit Card,0,0,27.640,27.64
3,410,19,51,88.61,4368.39,One Year,Fiber,0,UPI,0,0,84.008,84.01
4,664,56,47,141.36,6544.26,Month-to-Month,DSL,1,Bank Transfer,1,0,136.339,136.34
5,683,30,35,113.59,3922.95,One Year,Fiber,3,UPI,0,0,108.971,108.97
6,720,67,64,14.85,967.41,Month-to-Month,Fiber,4,Cash,0,1,14.883,14.88
7,861,51,14,138.33,2099.37,Month-to-Month,DSL,0,UPI,1,0,139.958,139.96
8,129,65,25,15.06,353.80,Month-to-Month,DSL,1,Bank Transfer,1,1,13.608,13.61
9,166,40,8,78.19,609.75,Month-to-Month,Fiber,5,Bank Transfer,1,0,67.750,67.75


In [30]:
# Percentage Column: Support tickets per tenure month
df_pandas['Support_ticket_per_month']=(df_pandas['support_tickets']/(df_pandas['tenure_months']+1)).round(2)
df_pandas.head(10)

,customer_id,age,tenure_months,monthly_charges,total_charges,contract_type,internet_service,support_tickets,payment_method,churn,is_senior,charge_per_tenure,Average_monthly_charge,Support_ticket_per_month
0,1,56,15,59.23,929.62,Month-to-Month,Fiber,5,UPI,1,0,58.101,58.10,0.31
1,182,23,38,80.86,3246.77,Month-to-Month,DSL,3,Cash,1,0,83.251,83.25,0.08
2,199,20,40,29.91,1133.23,Two Year,Fiber,3,Credit Card,0,0,27.640,27.64,0.07
3,410,19,51,88.61,4368.39,One Year,Fiber,0,UPI,0,0,84.008,84.01,0.00
4,664,56,47,141.36,6544.26,Month-to-Month,DSL,1,Bank Transfer,1,0,136.339,136.34,0.02
5,683,30,35,113.59,3922.95,One Year,Fiber,3,UPI,0,0,108.971,108.97,0.08
6,720,67,64,14.85,967.41,Month-to-Month,Fiber,4,Cash,0,1,14.883,14.88,0.06
7,861,51,14,138.33,2099.37,Month-to-Month,DSL,0,UPI,1,0,139.958,139.96,0.00
8,129,65,25,15.06,353.80,Month-to-Month,DSL,1,Bank Transfer,1,1,13.608,13.61,0.04
9,166,40,8,78.19,609.75,Month-to-Month,Fiber,5,Bank Transfer,1,0,67.750,67.75,0.56


In [31]:
# Growth Metric: Ratio of current monthly charge to historical average monthly charge
historical_avg=df_pandas['total_charges']/(df_pandas['tenure_months']+1)
df_pandas['spending_growth_metric']=(df_pandas['monthly_charges']/historical_avg).round(2)
df_pandas.head(10)

,customer_id,age,tenure_months,monthly_charges,total_charges,contract_type,internet_service,support_tickets,payment_method,churn,is_senior,charge_per_tenure,Average_monthly_charge,Support_ticket_per_month,spending_growth_metric
0,1,56,15,59.23,929.62,Month-to-Month,Fiber,5,UPI,1,0,58.101,58.10,0.31,1.02
1,182,23,38,80.86,3246.77,Month-to-Month,DSL,3,Cash,1,0,83.251,83.25,0.08,0.97
2,199,20,40,29.91,1133.23,Two Year,Fiber,3,Credit Card,0,0,27.640,27.64,0.07,1.08
3,410,19,51,88.61,4368.39,One Year,Fiber,0,UPI,0,0,84.008,84.01,0.00,1.05
4,664,56,47,141.36,6544.26,Month-to-Month,DSL,1,Bank Transfer,1,0,136.339,136.34,0.02,1.04
5,683,30,35,113.59,3922.95,One Year,Fiber,3,UPI,0,0,108.971,108.97,0.08,1.04
6,720,67,64,14.85,967.41,Month-to-Month,Fiber,4,Cash,0,1,14.883,14.88,0.06,1.00
7,861,51,14,138.33,2099.37,Month-to-Month,DSL,0,UPI,1,0,139.958,139.96,0.00,0.99
8,129,65,25,15.06,353.80,Month-to-Month,DSL,1,Bank Transfer,1,1,13.608,13.61,0.04,1.11
9,166,40,8,78.19,609.75,Month-to-Month,Fiber,5,Bank Transfer,1,0,67.750,67.75,0.56,1.15


In [32]:
# Convert back to Spark
df_spark=spark.createDataFrame(df_pandas)
df_spark.show()

+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+---------+-----------------+----------------------+------------------------+----------------------+
|customer_id|age|tenure_months|monthly_charges|total_charges| contract_type|internet_service|support_tickets|payment_method|churn|is_senior|charge_per_tenure|Average_monthly_charge|Support_ticket_per_month|spending_growth_metric|
+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+---------+-----------------+----------------------+------------------------+----------------------+
|          1| 56|           15|          59.23|       929.62|Month-to-Month|           Fiber|              5|           UPI|    1|        0|           58.101|                  58.1|                    0.31|                  1.02|
|        182| 23|           38|          80.86|      3246.77|Month-to-Month|    

# Task 5: Spark SQL

In [65]:
#Create Temporary Views.
df_silver.createOrReplaceTempView('view_1')

In [ ]:
#Query 1 Top Categories
spark.sql("SELECT contract_type, COUNT(customer_id) AS customer_count FROM view_1 GROUP BY contract_type").show()

+--------------+--------------+
| contract_type|customer_count|
+--------------+--------------+
|Month-to-Month|           583|
|      One Year|           278|
|      Two Year|           139|
+--------------+--------------+



In [70]:
# Query 2: Average Monthly Charges by Contract Type
spark.sql("SELECT contract_type ,AVG(monthly_charges) as avg_charges FROM view_1 GROUP BY contract_type").show()

+--------------+-----------------+
| contract_type|      avg_charges|
+--------------+-----------------+
|Month-to-Month|80.42109777015438|
|      One Year|80.04780575539571|
|      Two Year|77.90187050359717|
+--------------+-----------------+



In [72]:
# Query 3: Top 10 Customers with highest Total Charges
spark.sql("SELECT customer_id ,total_charges FROM view_1 ORDER BY total_charges DESC LIMIT 10").show()

+-----------+-------------+
|customer_id|total_charges|
+-----------+-------------+
|         95|     10772.52|
|        391|     10558.95|
|        179|     10445.05|
|        519|     10071.46|
|        940|      9940.84|
|        570|      9754.19|
|        939|      9571.84|
|        950|      9465.68|
|        501|      9457.37|
|        481|      9261.92|
+-----------+-------------+



In [78]:
#Query 4 Monthly Trend Analysis
spark.sql("""
    SELECT 
    CASE 
        WHEN tenure_months <= 6 THEN '0-6 Months (New)'
        WHEN tenure_months <= 12 THEN '7-12 Months (Infant)'
        WHEN tenure_months <= 24 THEN '13-24 Months (Mid-Tier)'
        ELSE '24+ Months (Loyal)'
    END AS tenure_bucket,
    COUNT(customer_id) AS total_cohort_customers,
    ROUND(AVG(monthly_charges), 2) AS cohort_avg_monthly_charge,
    ROUND(SUM(total_charges), 2) AS cohort_total_revenue
FROM view_1
GROUP BY 1
ORDER BY MIN(tenure_months) ASC
""").show()


+--------------------+----------------------+-------------------------+--------------------+
|       tenure_bucket|total_cohort_customers|cohort_avg_monthly_charge|cohort_total_revenue|
+--------------------+----------------------+-------------------------+--------------------+
|    0-6 Months (New)|                    89|                     79.9|            24439.03|
|7-12 Months (Infant)|                    85|                    81.32|            66208.37|
|13-24 Months (Mid...|                   162|                    83.87|           244614.89|
|  24+ Months (Loyal)|                   664|                    78.85|          2464973.09|
+--------------------+----------------------+-------------------------+--------------------+



In [82]:
#Query 5 Top 10 Records by Business Metric

spark.sql("""
SELECT 
    customer_id,
    contract_type,
    internet_service,
    tenure_months,
    monthly_charges,
    ROUND(total_charges, 2) AS lifetime_value
FROM view_1
WHERE churn = 0  
ORDER BY total_charges DESC
LIMIT 10
""").show()

+-----------+--------------+----------------+-------------+---------------+--------------+
|customer_id| contract_type|internet_service|tenure_months|monthly_charges|lifetime_value|
+-----------+--------------+----------------+-------------+---------------+--------------+
|        391|      One Year|           Fiber|           69|         149.49|      10558.95|
|        179|Month-to-Month|             DSL|           69|         145.22|      10445.05|
|        950|      One Year|            None|           70|         144.96|       9465.68|
|        501|      One Year|             DSL|           65|         141.16|       9457.37|
|        552|Month-to-Month|             DSL|           67|          132.9|        9145.0|
|        987|      One Year|             DSL|           70|         128.58|       9102.87|
|        723|      One Year|           Fiber|           64|          129.8|       9018.13|
|        804|      One Year|            None|           58|         143.71|       8931.79|

# Task 6: Advanced Transformations


In [35]:
df_silver.show()

+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+---------+-----------------+
|customer_id|age|tenure_months|monthly_charges|total_charges| contract_type|internet_service|support_tickets|payment_method|churn|is_senior|charge_per_tenure|
+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+---------+-----------------+
|          1| 56|           15|          59.23|       929.62|Month-to-Month|           Fiber|              5|           UPI|    1|        0|           58.101|
|        182| 23|           38|          80.86|      3246.77|Month-to-Month|             DSL|              3|          Cash|    1|        0|           83.251|
|        199| 20|           40|          29.91|      1133.23|      Two Year|           Fiber|              3|   Credit Card|    0|        0|            27.64|
|        410| 19|           51|          88.61

In [ ]:
#row_number()
#rank()
#dense_rank()
#lag()
#lead()

df_window=df_silver.withColumn("row_number",row_number().over(Window.orderBy(col('internet_service'))))\
    .withColumn("rank",rank().over(Window.orderBy(col('internet_service').asc())))\
        .withColumn("dense_rank",dense_rank().over(Window.orderBy(col('internet_service').asc())))\
            .withColumn("lag",lag('internet_service').over(Window.orderBy(col('internet_service').asc())))\
                .withColumn("lead",lead('internet_service').over(Window.orderBy(col('internet_service').asc()))) # type: ignore
    
df_window.show(50)

+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+---------+-----------------+----------+----+----------+----+----+
|customer_id|age|tenure_months|monthly_charges|total_charges| contract_type|internet_service|support_tickets|payment_method|churn|is_senior|charge_per_tenure|row_number|rank|dense_rank| lag|lead|
+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+---------+-----------------+----------+----+----------+----+----+
|        182| 23|           38|          80.86|      3246.77|Month-to-Month|             DSL|              3|          Cash|    1|        0|           83.251|         1|   1|         1|NULL| DSL|
|        664| 56|           47|         141.36|      6544.26|Month-to-Month|             DSL|              1| Bank Transfer|    1|        0|          136.339|         2|   1|         1| DSL| DSL|
|        861| 51|   

In [37]:
# Join
lookup_data=[("Fiber", "High_Speed"), ("DSL", "Medium_Speed"), ("None", "No_Internet")]
lookup_df = spark.createDataFrame(lookup_data, ["internet_service", "speed_category"])

#Inner Join
df_inner_join=df_silver.join(lookup_df,df_silver['internet_service']== lookup_df["internet_service"],'inner')
df_inner_join.show()


+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+---------+-----------------+----------------+--------------+
|customer_id|age|tenure_months|monthly_charges|total_charges| contract_type|internet_service|support_tickets|payment_method|churn|is_senior|charge_per_tenure|internet_service|speed_category|
+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+---------+-----------------+----------------+--------------+
|        167| 56|            5|          94.88|       517.19|Month-to-Month|           Fiber|              0|           UPI|    0|        0|           86.198|           Fiber|    High_Speed|
|        876| 38|           44|          13.56|       619.29|      One Year|           Fiber|              0|   Credit Card|    0|        0|           13.762|           Fiber|    High_Speed|
|        811| 66|           60|          94.5

In [38]:
#Left Join
df_left_join=df_silver.join(lookup_df,df_silver['internet_service']== lookup_df["internet_service"],'left')
df_left_join.show()

+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+---------+-----------------+----------------+--------------+
|customer_id|age|tenure_months|monthly_charges|total_charges| contract_type|internet_service|support_tickets|payment_method|churn|is_senior|charge_per_tenure|internet_service|speed_category|
+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+---------+-----------------+----------------+--------------+
|        182| 23|           38|          80.86|      3246.77|Month-to-Month|             DSL|              3|          Cash|    1|        0|           83.251|             DSL|  Medium_Speed|
|        664| 56|           47|         141.36|      6544.26|Month-to-Month|             DSL|              1| Bank Transfer|    1|        0|          136.339|             DSL|  Medium_Speed|
|        861| 51|           14|         138.3

## Task 7: Spark ML Pipeline


In [ ]:
from pyspark.ml import Pipeline# type:ignore
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler# type:ignore
from pyspark.ml.classification import LogisticRegression# type:ignore

In [85]:
cat_cols = ['contract_type', 'internet_service', 'payment_method']
num_cols = ['age', 'tenure_months', 'monthly_charges', 'total_charges', 'support_tickets', 'is_senior', 'charge_per_tenure']
df_ml = df_silver.select(cat_cols + num_cols + ["churn"]).dropna()

In [91]:
# String Indexer
indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_index", handleInvalid="keep") for c in cat_cols]
indexers

[StringIndexer_bf863f39a3d1,
 StringIndexer_873222f91a16,
 StringIndexer_68b65c69e75d]

In [92]:
#OneHotEncoder
encoders = [OneHotEncoder(inputCol=f"{c}_index", outputCol=f"{c}_vec") for c in cat_cols]
encoders

[OneHotEncoder_4b7f6f27b340,
 OneHotEncoder_bc63be904154,
 OneHotEncoder_3240031926d3]

In [94]:
#VectorAssembler
assembler = VectorAssembler(
    inputCols=[f"{c}_vec" for c in cat_cols] + num_cols, 
    outputCol="features"
)
assembler

VectorAssembler_4e7517da9a13

In [ ]:
#Logistic Regression
lr = LogisticRegression(featuresCol="features", labelCol="churn",predictionCol="prediction")
lr

LogisticRegression_b9e9465dc8e6

In [ ]:
# Pipeline
pipeline = Pipeline(stages=indexers + encoders + [assembler, lr])
pipeline

Pipeline_5f671cb2f4e2

In [104]:
# Train Test Split
train_data, test_data = df_ml.randomSplit([0.8, 0.2], seed=42)
train_data.head(5)
test_data.head(5)

[Row(contract_type='Month-to-Month', internet_service='DSL', payment_method='Bank Transfer', age=21, tenure_months=56, monthly_charges=28.55, total_charges=1477.38, support_tickets=3, is_senior=0, charge_per_tenure=25.919, churn=1),
 Row(contract_type='Month-to-Month', internet_service='DSL', payment_method='Bank Transfer', age=26, tenure_months=19, monthly_charges=142.17, total_charges=2641.41, support_tickets=1, is_senior=0, charge_per_tenure=132.07, churn=1),
 Row(contract_type='Month-to-Month', internet_service='DSL', payment_method='Bank Transfer', age=26, tenure_months=66, monthly_charges=40.34, total_charges=2584.42, support_tickets=2, is_senior=0, charge_per_tenure=38.573, churn=1),
 Row(contract_type='Month-to-Month', internet_service='DSL', payment_method='Bank Transfer', age=31, tenure_months=9, monthly_charges=111.99, total_charges=967.33, support_tickets=0, is_senior=0, charge_per_tenure=96.733, churn=1),
 Row(contract_type='Month-to-Month', internet_service='DSL', payment

In [107]:
# Fit Model
print("Training Model...")
model = pipeline.fit(train_data)
predictions = model.transform(test_data)
predictions.head(5)

Training Model...


[Row(contract_type='Month-to-Month', internet_service='DSL', payment_method='Bank Transfer', age=21, tenure_months=56, monthly_charges=28.55, total_charges=1477.38, support_tickets=3, is_senior=0, charge_per_tenure=25.919, churn=1, contract_type_index=0.0, internet_service_index=1.0, payment_method_index=3.0, contract_type_vec=SparseVector(3, {0: 1.0}), internet_service_vec=SparseVector(3, {1: 1.0}), payment_method_vec=SparseVector(4, {3: 1.0}), features=SparseVector(17, {0: 1.0, 4: 1.0, 9: 1.0, 10: 21.0, 11: 56.0, 12: 28.55, 13: 1477.38, 14: 3.0, 16: 25.919}), rawPrediction=DenseVector([-0.1909, 0.1909]), probability=DenseVector([0.4524, 0.5476]), prediction=1.0),
 Row(contract_type='Month-to-Month', internet_service='DSL', payment_method='Bank Transfer', age=26, tenure_months=19, monthly_charges=142.17, total_charges=2641.41, support_tickets=1, is_senior=0, charge_per_tenure=132.07, churn=1, contract_type_index=0.0, internet_service_index=1.0, payment_method_index=3.0, contract_type_

In [ ]:
# Show a quick preview of actual vs predicted churn values
predictions.select("churn", "prediction", "probability").show(5)

+-----+----------+--------------------+
|churn|prediction|         probability|
+-----+----------+--------------------+
|    1|       1.0|[0.45242078060636...|
|    1|       1.0|[0.17535136067061...|
|    1|       0.0|[0.54574707059686...|
|    1|       1.0|[0.25419321802762...|
|    1|       1.0|[0.23208728630818...|
+-----+----------+--------------------+
only showing top 5 rows


# Task 8: Performance Optimization

In [39]:
import time
start_time=time.time()
unoptimized_agg_time = time.time() - start_time

In [51]:
# Unoptimized Operation (Directly from File)
start_time = time.time()
spark.read.parquet(f"{base_path}/silver/cleaned_data.parquet").groupBy("contract_type").agg(avg("monthly_charges")).collect()
unoptimized_agg_time = time.time() - start_time
unoptimized_agg_time

0.3079662322998047

In [52]:
#Cache
df_opt = spark.read.parquet(f"{base_path}/silver/cleaned_data.parquet")
df_opt.cache()
df_opt.count()
start_time = time.time()
optimized_agg_time = time.time() - start_time
optimized_agg_time


0.00011181831359863281

In [55]:
# Optimization 2: Repartitioning
df_repartitioned = df_opt.repartition(8)
df_repartitioned.show()

+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+---------+-----------------+
|customer_id|age|tenure_months|monthly_charges|total_charges| contract_type|internet_service|support_tickets|payment_method|churn|is_senior|charge_per_tenure|
+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+---------+-----------------+
|        789| 65|           70|          81.04|      6004.05|      One Year|           Fiber|              2|   Credit Card|    0|        1|           84.564|
|        234| 49|           54|         122.22|      6944.08|Month-to-Month|           Fiber|              3|   Credit Card|    0|        0|          126.256|
|        110| 41|           60|          58.66|      3681.28|      One Year|           Fiber|              1| Bank Transfer|    0|        0|           60.349|
|        612| 53|           13|          83.84

In [ ]:
#Optimization 3: Broadcast Join
start_time = time.time()
df_opt.join(broadcast(lookup_df), "internet_service").count()#type:ignore 
broadcast_join_time = time.time() - start_time
broadcast_join_time

0.5901937484741211

In [88]:
print(f"Unoptimized Aggregation Time: {unoptimized_agg_time:.4f} seconds")
print(f"Optimized (Cached) Aggregation Time: {optimized_agg_time:.4f} seconds")
print(f"Broadcast Join Time: {broadcast_join_time:.4f} seconds")

Unoptimized Aggregation Time: 0.3080 seconds
Optimized (Cached) Aggregation Time: 0.0001 seconds
Broadcast Join Time: 0.5902 seconds
